In [1]:
!pip install -q trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 14.0 MB/s eta 0:00:0000:01


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

In [3]:
from huggingface_hub import login
login("make your own"
      "")

In [4]:
import torch
from peft import PeftModel
import gc# # Load LoRA adapter and merge
# peft_model = PeftModel.from_pretrained(base_model_eval, './tunned_model')
# merged_model = peft_model.merge_and_unload()  # Merges LoRA into base weights
# merged_model.eval()

# # This is now our inference model — replace the variable name for clarity
# model = merged_model

# print(f'✅ Fine-tuned model ready for inference.')
# print(f'   VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

# Free trainer memory
torch.cuda.empty_cache()
gc.collect()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Merging LoRA weights into base model for inference...')

# Reload base model fresh
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    dtype=torch.float16,
    device_map=device,
)



Merging LoRA weights into base model for inference...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [6]:
import re
from datasets import load_dataset

print('Loading GSM8K train split...')
gsm8k_test = load_dataset('gsm8k', 'main', split='test')
print(f'Test size:  {len(gsm8k_test)}')

Loading GSM8K train split...


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Test size:  1319


In [7]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

In [8]:
import re
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════
# GENERATION
# ═══════════════════════════════════════════════════════════

def generate_text(prompt, max_new_tokens=512, temperature=0.0, top_p=0.95):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model.generate( # CHANGE
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature if temperature > 0 else 1.0,
            top_p=top_p,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )

# ═══════════════════════════════════════════════════════════
# ANSWER EXTRACTION — robust multi-pattern
# ═══════════════════════════════════════════════════════════

def clean_number(s):
    return s.strip().rstrip('.,;:').replace(',', '').replace('$', '').replace('%', '')

def extract_answer(text):
    if not text:
        return None
    patterns = [
        r'(?:Answer|answer|ANSWER)\s*[:=]\s*\$?([\-0-9,\.]+)',
        r'(?:The answer is|Therefore,?|Thus,?)\s*\$?([\-0-9,\.]+)',
        r'####\s*([\-0-9,\.]+)',
    ]
    for pat in patterns:
        m = re.search(pat, text)
        if m:
            val = clean_number(m.group(1))
            if val:
                return val
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    for line in reversed(lines):
        if re.match(r'^\$?[\-0-9,\.]+$', line):
            val = clean_number(line)
            if val:
                return val
    last_text = ' '.join(lines[-3:]) if len(lines) >= 3 else text
    numbers = re.findall(r'-?\d+\.?\d*', last_text)
    return clean_number(numbers[-1]) if numbers else None

def get_ground_truth(answer_text):
    nums = re.findall(r'####\s*([\-0-9,\.]+)', answer_text)
    if nums:
        return clean_number(nums[-1])
    nums = re.findall(r'-?\d+\.?\d*', answer_text)
    return clean_number(nums[-1]) if nums else None

def answers_match(pred, gold):
    if pred is None or gold is None:
        return False
    try:
        return abs(float(pred) - float(gold)) < 0.01
    except:
        return pred.strip() == gold.strip()

# ═══════════════════════════════════════════════════════════
# ARITHMETIC VERIFICATION
# ═══════════════════════════════════════════════════════════

def verify_arithmetic(text):
    equations = re.findall(
        r'([\d\.\+\-\*/\(\)\s]+)\s*=\s*([\d\.\+\-\*/\(\)\s]+)', text
    )
    if not equations:
        return 0.5
    correct, total = 0, 0
    for left, right in equations:
        try:
            if abs(eval(left.strip()) - eval(right.strip())) <= 0.01:
                correct += 1
            total += 1
        except:
            continue
    return correct / total if total > 0 else 0.5

# ═══════════════════════════════════════════════════════════
# EMBEDDING (for PRM classifier)
# ═══════════════════════════════════════════════════════════

def get_embedding(text, max_length=128):
    inputs = tokenizer(
        text, return_tensors='pt', truncation=True,
        max_length=max_length, padding=True
    ).to(device)
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True)
    hidden = out.hidden_states[-1]
    mask = inputs['attention_mask'].unsqueeze(-1).float()
    return ((hidden * mask).sum(1) / mask.sum(1)).squeeze(0).cpu().float().numpy()

print('✅ Helper functions loaded.')

✅ Helper functions loaded.


In [9]:
gsm8k_eval = gsm8k_test.select(range(50))
print(len(gsm8k_eval))

50


In [10]:
BASELINE_PROMPT = """Solve this math problem. Give only the final numeric answer.

Problem: {problem}

Answer:"""

results_baseline = []
print(f'Running zero-shot baseline on fine-tuned model ({20} problems)...')

for i, example in enumerate(tqdm(gsm8k_eval, desc='FT Baseline')):
    problem = example['question']
    correct_answer = get_ground_truth(example['answer'])
    generated = generate_text(BASELINE_PROMPT.format(problem=problem), temperature=0.0)
    pred_answer = extract_answer(generated)
    print(f"{pred_answer} {"="*10} {correct_answer}")
    is_correct = answers_match(pred_answer, correct_answer)
    results_baseline.append({
        'problem_id': i, 'problem': problem, 'generated': generated,
        'pred_answer': pred_answer, 'correct_answer': correct_answer, 'is_correct': is_correct,
    })
    if (i + 1) % 5 == 0:
        acc = sum(r['is_correct'] for r in results_baseline) / len(results_baseline)
        print(f'  [{i+1}/{50}] Accuracy: {acc:.1%}')

ft_baseline_acc = sum(r['is_correct'] for r in results_baseline) / len(results_baseline)
print(f'FT Zero-Shot Baseline: {ft_baseline_acc:.1%}')

Running zero-shot baseline on fine-tuned model (20 problems)...


FT Baseline:   0%|          | 0/50 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


66.67 ========== 18
3 ========== 3
000 ========== 70000
4 ========== 540
20 ========== 20
  [5/50] Accuracy: 40.0%
16 ========== 64
260 ========== 260
120 ========== 160
315 ========== 45
50 ========== 460
  [10/50] Accuracy: 30.0%
366 ========== 366
694 ========== 694
8 ========== 13
18 ========== 18
60 ========== 60
  [15/50] Accuracy: 46.7%
0 ========== 125
230 ========== 230
500 ========== 57500
21 ========== 7
6 ========== 6
  [20/50] Accuracy: 45.0%
15 ========== 15
14 ========== 14
8 ========== 7
10 ========== 8
26.00 ========== 26
  [25/50] Accuracy: 48.0%
2 ========== 2
243 ========== 243
16 ========== 16
20 ========== 25
104 ========== 104
  [30/50] Accuracy: 53.3%
10 ========== 109
80 ========== 80
8 ========== 35
70 ========== 70
23 ========== 23
  [35/50] Accuracy: 54.3%
60 ========== 9
10 ========== 75
12 ========== 2
10 ========== 10
18 ========== 18
  [40/50] Accuracy: 52.5%
8 ========== 8
200 ========== 200
6 ========== 26
0 ========== 48
10 ========== 20
  [45/50] Acc

In [11]:
# gemma 2b

# Running zero-shot baseline on fine-tuned model (20 problems)...
# FT Baseline: 100%
#  50/50 [00:18<00:00,  3.31it/s]
# None ========== 18
# 3 ========== 3
# 100000.0 ========== 70000
# 1800 ========== 540
# 10 ========== 20
#   [5/50] Accuracy: 20.0%
# 1000.00 ========== 64
# None ========== 260
# 100.00 ========== 160
# 120.00 ========== 45
# 1000.00 ========== 460
#   [10/50] Accuracy: 10.0%
# 180 ========== 366
# 220 ========== 694
# 6.00 ========== 13
# 15 ========== 18
# 10.00 ========== 60
#   [15/50] Accuracy: 6.7%
# 520.00 ========== 125
# 160 ========== 230
# 10500 ========== 57500
# 12 ========== 7
# 6.00 ========== 6
#   [20/50] Accuracy: 10.0%
# 24.00 ========== 15
# 10 ========== 14
# 15 ========== 7
# 10.0 ========== 8
# 78.00 ========== 26
#   [25/50] Accuracy: 8.0%
# 1.0 ========== 2
# 108.00 ========== 243
# 240.00 ========== 16
# 35 ========== 25
# 100.00 ========== 104
#   [30/50] Accuracy: 6.7%
# 31 ========== 109
# 70.00 ========== 80
# 50 ========== 35
# 40 ========== 70
# 15 ========== 23
#   [35/50] Accuracy: 5.7%
# 75 ========== 9
# 150.00 ========== 75
# 13 ========== 2
# 8.33 ========== 10
# None ========== 18
#   [40/50] Accuracy: 5.0%
# 4 ========== 8
# 1200 ========== 200
# 14 ========== 26
# 100.00 ========== 48
# 10.00 ========== 20
#   [45/50] Accuracy: 4.4%
# 48 ========== 104
# 19 ========== 163
# 200.00 ========== 800
# None ========== 8
# 12 ========== 30
#   [50/50] Accuracy: 4.0%
# FT Zero-Shot Baseline: 4.0%

In [12]:
# Qwen/Qwen2.5-7B-Instruct


#  Running zero-shot baseline on fine-tuned model (20 problems)...
# FT Baseline: 100%
#  50/50 [16:35<00:00, 19.25s/it]
# The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
# 8 ========== 18
# 3 ========== 3
# 70000 ========== 70000
# 540 ========== 540
# 10 ========== 20
#   [5/50] Accuracy: 60.0%
# 72 ========== 64
# 260 ========== 260
# 120 ========== 160
# 190 ========== 45
# 480 ========== 460
#   [10/50] Accuracy: 40.0%
# 270 ========== 366
# 418 ========== 694
# 6 ========== 13
# 18 ========== 18
# 36 ========== 60
#   [15/50] Accuracy: 33.3%
# 125 ========== 125
# 230 ========== 230
# 47500 ========== 57500
# 7 ========== 7
# 6 ========== 6
#   [20/50] Accuracy: 45.0%
# 13 ========== 15
# 8 ========== 14
# 9 ========== 7
# 8 ========== 8
# 26 ========== 26
#   [25/50] Accuracy: 44.0%
# 2 ========== 2
# 189 ========== 243
# 16 ========== 16
# 35 ========== 25
# 95 ========== 104
#   [30/50] Accuracy: 43.3%
# 10 ========== 109
# 76 ========== 80
# 28 ========== 35
# 65 ========== 70
# 23 ========== 23
#   [35/50] Accuracy: 40.0%
# 14 ========== 9
# 75 ========== 75
# 13 ========== 2
# 10 ========== 10
# 18 ========== 18
#   [40/50] Accuracy: 42.5%
# 8 ========== 8
# 700 ========== 200
# 34 ========== 26
# 100 ========== 48
# 20 ========== 20
#   [45/50] Accuracy: 42.2%
# 68 ========== 104
# 151 ========== 163
# 800 ========== 800
# 4 ========== 8
# 120 ========== 30
#   [50/50] Accuracy: 40.0%
# FT Zero-Shot Baseline: 40.0%

In [13]:
# mistralai/Mistral-7B-Instruct-v0.3


# unning zero-shot baseline on fine-tuned model (20 problems)...
# FT Baseline: 100%
#  50/50 [08:49<00:00, 16.20s/it]
# The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
# 18 ========== 18
# 4 ========== 3
# 100000 ========== 70000
# 5400 ========== 540
# 100 ========== 20
#   [5/50] Accuracy: 20.0%
# 120 ========== 64
# 160 ========== 260
# 40 ========== 160
# 11 ========== 45
# 470 ========== 460
#   [10/50] Accuracy: 10.0%
# 150 ========== 366
# 1348 ========== 694
# 10 ========== 13
# 20 ========== 18
# 60 ========== 60
#   [15/50] Accuracy: 13.3%
# 29 ========== 125
# 230 ========== 230
# 17000 ========== 57500
# 12 ========== 7
# 6 ========== 6
#   [20/50] Accuracy: 20.0%
# 16 ========== 15
# 17 ========== 14
# 7 ========== 7
# 00 ========== 8
# 26.00 ========== 26
#   [25/50] Accuracy: 24.0%
# 3 ========== 2
# 246.00 ========== 243
# 120.00 ========== 16
# 35 ========== 25
# 66 ========== 104
#   [30/50] Accuracy: 20.0%
# 42 ========== 109
# 88.5 ========== 80
# 50 ========== 35
# 60 ========== 70
# 30 ========== 23
#   [35/50] Accuracy: 17.1%
# 32 ========== 9
# 30.00 ========== 75
# 0 ========== 2
# 7 ========== 10
# 3 ========== 18
#   [40/50] Accuracy: 15.0%
# 16 ========== 8
# 1200 ========== 200
# 60 ========== 26
# 150 ========== 48
# 40.00 ========== 20
#   [45/50] Accuracy: 13.3%
# 4 ========== 104
# 107 ========== 163
# 320 ========== 800
# 1 ========== 8
# 120 ========== 30
#   [50/50] Accuracy: 12.0%
# FT Zero-Shot Baseline: 12.0%

In [ ]:
# microsoft/phi-4k-it


# Running zero-shot baseline on fine-tuned model (20 problems)...
# FT Baseline: 100%
#  50/50 [11:24<00:00, 15.04s/it]
# The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
# 66.67 ========== 18
# 3 ========== 3
# 000 ========== 70000
# 4 ========== 540
# 20 ========== 20
#   [5/50] Accuracy: 40.0%
# 16 ========== 64
# 260 ========== 260
# 120 ========== 160
# 45 ========== 45
# 50 ========== 460
#   [10/50] Accuracy: 40.0%
# 366 ========== 366
# 694 ========== 694
# 12 ========== 13
# 18 ========== 18
# 60 ========== 60
#   [15/50] Accuracy: 53.3%
# 0 ========== 125
# 230 ========== 230
# 500 ========== 57500
# 21 ========== 7
# 6 ========== 6
#   [20/50] Accuracy: 50.0%
# 15 ========== 15
# 14 ========== 14
# 8 ========== 7
# 10 ========== 8
# 26.00 ========== 26
#   [25/50] Accuracy: 52.0%
# 2 ========== 2
# 243 ========== 243
# 16 ========== 16
# 50 ========== 25
# 104 ========== 104
#   [30/50] Accuracy: 56.7%
# 10 ========== 109
# 80 ========== 80
# 8 ========== 35
# 70 ========== 70
# 23 ========== 23
#   [35/50] Accuracy: 57.1%
# 9 ========== 9
# 38 ========== 75
# 15 ========== 2
# 10 ========== 10
# 18 ========== 18
#   [40/50] Accuracy: 57.5%
# 8 ========== 8
# 200 ========== 200
# 6 ========== 26
# 0 ========== 48
# 10 ========== 20
#   [45/50] Accuracy: 55.6%
# None ========== 104
# None ========== 163
# 120 ========== 800
# 8 ========== 8
# 30 ========== 30
#   [50/50] Accuracy: 54.0%
# FT Zero-Shot Baseline: 54.0%

In [15]:
# microsoft/Phi-3-mini-4k-instruct + my tunned weight


#  Running zero-shot baseline on fine-tuned model (20 problems)...
# FT Baseline: 100% 50/50 [04:24<00:00,  5.08s/it]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
# 18 ========== 18
# 3 ========== 3
# 000 ========== 70000
# 540 ========== 540
# 21 ========== 20
#   [5/50] Accuracy: 60.0%
# 64 ========== 64
# 260 ========== 260
# 80 ========== 160
# 135 ========== 45
# 460 ========== 460
#   [10/50] Accuracy: 60.0%
# 366 ========== 366
# 694 ========== 694
# 12 ========== 13
# 11 ========== 18
# 60 ========== 60
#   [15/50] Accuracy: 60.0%
# 29 ========== 125
# 230 ========== 230
# 57500 ========== 57500
# 7 ========== 7
# 1.5 ========== 6
#   [20/50] Accuracy: 60.0%
# 0.61 ========== 15
# 2 ========== 14
# 7 ========== 7
# 10 ========== 8
# 26 ========== 26
#   [25/50] Accuracy: 56.0%
# 2 ========== 2
# 243 ========== 243
# 16 ========== 16
# 25 ========== 25
# 104 ========== 104
#   [30/50] Accuracy: 63.3%
# 109 ========== 109
# 80 ========== 80
# 35 ========== 35
# 70 ========== 70
# 23 ========== 23
#   [35/50] Accuracy: 68.6%
# 9 ========== 9
# 30 ========== 75
# 11 ========== 2
# 10 ========== 10
# 36 ========== 18
#   [40/50] Accuracy: 65.0%
# 8 ========== 8
# 200 ========== 200
# 26 ========== 26
# 48 ========== 48
# 20 ========== 20
#   [45/50] Accuracy: 68.9%
# 104 ========== 104
# 163 ========== 163
# 800 ========== 800
# 8 ========== 8
# 30 ========== 30
#   [50/50] Accuracy: 72.0%
# FT Zero-Shot Baseline: 72.0%


In [14]:
# 5 shot + microsoft/Phi-3-mini-4k-instruct + my tunned weight

#  Running 5-shot CoT on fine-tuned model (50 problems)...
# 5-Shot CoT: 100% 50/50 [07:13<00:00,  3.39s/it]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
# 18 ========== 18
# 3 ========== 3
# 70000 ========== 70000
# 540 ========== 540
# 2 ========== 20
#   [5/50] Accuracy: 80.0%
# 64 ========== 64
# 260 ========== 260
# 60 ========== 160
# 235 ========== 45
# 460 ========== 460
#   [10/50] Accuracy: 70.0%
# 366 ========== 366
# 694 ========== 694
# 12 ========== 13
# 18 ========== 18
# 60 ========== 60
#   [15/50] Accuracy: 73.3%
# 29 ========== 125
# 230 ========== 230
# 57500 ========== 57500
# 7 ========== 7
# 0.6 ========== 6
#   [20/50] Accuracy: 70.0%
# 21.55 ========== 15
# -2 ========== 14
# 7 ========== 7
# 8 ========== 8
# 26 ========== 26
#   [25/50] Accuracy: 68.0%
# 2 ========== 2
# 243 ========== 243
# 16 ========== 16
# 25 ========== 25
# 104 ========== 104
#   [30/50] Accuracy: 73.3%
# 109 ========== 109
# 80 ========== 80
# 35 ========== 35
# 70 ========== 70
# 23 ========== 23
#   [35/50] Accuracy: 77.1%
# 9 ========== 9
# 75 ========== 75
# 2.66666667 ========== 2
# 10 ========== 10
# 48 ========== 18
#   [40/50] Accuracy: 75.0%
# 8 ========== 8
# 200 ========== 200
# 26 ========== 26
# 48 ========== 48
# 20 ========== 20
#   [45/50] Accuracy: 77.8%
# 104 ========== 104
# 163 ========== 163
# 800 ========== 800
# 8 ========== 8
# 30 ========== 30
#   [50/50] Accuracy: 80.0%

# ==================================================
# 5-Shot CoT         : 80.0%